# Impedance ML Engineering — Session 10
## Control overfitting and debug neural training
**Core: 60 minutes · Neural networks · Optional extension: 20–40 minutes**

## Why this matters
How can you distinguish capacity, optimization, preprocessing, and evaluation problems?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Use tiny-subset overfitting as a training diagnostic.
- Apply validation-based early stopping with safe checkpoint copies.
- Compare one regularization change at a time.
- Separate seed variability from patient-sampling uncertainty.

## R-to-Python / prerequisite bridge
`deepcopy(model.state_dict())` preserves a checkpoint; `.train()` and `.eval()` change dropout behavior; weight decay acts through the optimizer.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
Capacity, optimization, and data leakage create different failure patterns. A tiny-subset overfit test diagnoses whether a model/loop can fit; it does not evaluate usefulness. Regularization restricts effective capacity; weight decay penalizes large weights and dropout introduces random feature omission during training.

Early stopping selects an epoch using validation, so it is model selection. A copied checkpoint must preserve the actual tensors at the selected epoch. Restoring weights alone does not restore optimizer state for continued training, although it is enough for inference.

Imports: `import copy`. Use `copy.deepcopy(model.state_dict())`, `model.load_state_dict(best_state)`, `nn.Dropout(p=...)`, and optimizer `weight_decay=...`. Keep validation patients separate and freeze the stopping rule before comparing recipes.

## Task 1 — Recreate a trustworthy reference (6 min)
Use the Session 09 input recipe and a 16-unit MLP. Set seed 42 and record preprocessor, shape, parameters, learning rate, and validation MAE. Fresh-kernel setup must rebuild the model and optimizer. Predict target shape before loss.

## Task 2 — Overfit a tiny subset (8 min)
Select eight training rows, disable regularization, and train up to 200 steps. Plot loss. Use this only to diagnose whether the implementation can fit; do not score it as useful performance. If loss remains high, inspect scale, shapes, gradients, and learning rate before increasing capacity.

## Task 3 — Implement early stopping (10 min)
On the full training set, evaluate validation after each epoch, maximum 60 epochs, patience 8. Save `copy.deepcopy(model.state_dict())` only when validation MAE improves; restore it before reporting. Keep selection metric and stopping rule explicit. Explain why a shallow reference to state_dict can track later updates instead of freezing the best epoch.

## Task 4 — Compare a capacity change (7 min)
Compare one hidden layer of width 16 versus 64 using the same maximum epoch budget and stopping rule. Record train/validation gaps, best epoch, parameter count, and time. Do not use test to choose width. A larger model may lose.

## Task 5 — Add one regularizer (8 min)
Return to width 16 and compare Adam weight_decay=0 versus 1e-3 with all other choices fixed. Optionally replace this comparison with dropout p=0.2, but do not change both at once. Explain which mechanism you chose and verify train/eval behavior.

## Task 6 — Repeat the selected recipe (7 min)
Run seeds 42, 43, 44 on the same partitions and report all three validation MAEs. Distinguish initialization/minibatch variation from uncertainty over new patients. Do not keep only the winning seed.

## Task 7 — Build a debugging decision log (6 min)
For exploding loss, flat loss, large train/validation gap, and implausibly good validation, propose one diagnostic check each. Inspect gradient norms for one epoch and check input/target finiteness. Record what evidence would justify gradient clipping versus fixing a data bug.

## Task 8 — Write the experiment conclusion (0–2 min)
Choose a recipe for the next neural experiment based on validation evidence and simplicity. Record the search budget and decisions. State that reused validation performance is exploratory; test remains reserved.

## Retrieval practice
From memory, write why checkpoint copying, restoring the best state, eval mode, and no_grad are separate responsibilities.

## Professional reasoning
1. What could go wrong if validation is used to fit target scaling?
2. Why does a dropout model need eval mode when predicting?
3. Why is choosing the best random seed another form of model selection?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Use a learning-rate comparison of 0.001 versus 0.01 with the same budget and log gradients. Do not combine it with the regularization search.

## API references
[PyTorch training basics](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html) · [Conv2d reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)